In [ ]:
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
import os

def encrypt_blob(key: bytes, data: bytes):
    """
    Encrypt raw bytes using AES-GCM.

    Returns:
        nonce + ciphertext + tag (all needed for ESP32)
    """

    # AES-GCM standard nonce size = 12 bytes
    nonce = os.urandom(12)

    aesgcm = AESGCM(key)

    ciphertext = aesgcm.encrypt(nonce, data, None)

    # final packet = nonce + encrypted blob
    return nonce + ciphertext


key = AESGCM.generate_key(bit_length=256)

data = b"Command|"

encrypted = encrypt_blob(key, data)

print("Encrypted size:", len(encrypted))
print(data)
encrypted


In [ ]:
def decrypt_blob(key: bytes, packet: bytes):
    """
    Decrypts a packet created by encrypt_blob().

    Packet format:
        [12 bytes nonce][ciphertext + tag]
    """

    # Extract nonce (first 12 bytes)
    nonce = packet[:12]

    # Everything else is encrypted data + authentication tag
    ciphertext = packet[12:]

    aesgcm = AESGCM(key)

    # Decrypt (returns original plaintext bytes)
    plaintext = aesgcm.decrypt(nonce, ciphertext, None)

    return plaintext


decrypt_blob(key, encrypted)

In [ ]:
from Structures import *
import socket
import uuid


mac = uuid.getnode()

message = MessageUDP()
message.type = MessageTypes.command
message.mac_address = mac
message.device_name = socket.gethostname()
message.is_encrypted = False

command = Command()
command.type = CommandTypes.setting
command.sequence_number = 3003



command_setting = CommandSetting()
command_setting.brightness = 0.0
command_setting.transition_speed = 0.3


command.content = command_setting 
message.content = command


In [ ]:
message.__dict__

In [ ]:
message.content.__dict__

In [ ]:
message.content.content.__dict__

In [ ]:
message_bytes = message.to_bytes()

print(len(message_bytes))

In [ ]:
message_bytes

In [ ]:
arr = b""
arr += 0xff.to_bytes()

arr

In [ ]:
class Pet:
    name: str = "stary"
    happiness: float = 0.5


    def feed(self):
        print(self.happiness)


class Cat(Pet):
    def __init__(self):
        self.name: str = "mimi"


class Dog(Pet):
    def __init__(self):
        self.name: str = "spark"


cat = Cat()

print(cat.name)
print(cat.happiness)

In [2]:
from Structures import *
import socket
import uuid

# Create UDP socket
client_socket = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)

server_address = ('192.168.1.29', 39394)


def set_setting(value, transition_speed = 0.3):
    mac = uuid.getnode()

    message = MessageUDP()
    message.type = MessageTypes.command
    message.mac_address = mac
    message.device_name = socket.gethostname()
    message.is_encrypted = False

    command = Command()
    command.type = CommandTypes.setting
    command.sequence_number = 3003



    command_setting = CommandSetting()
    command_setting.brightness = int(255 * value)
    command_setting.transition_speed = transition_speed


    command.content = command_setting 
    message.content = command

    client_socket.sendto(message.to_bytes(), server_address)


def set_color(rgb_list):
    mac = uuid.getnode()


    message = MessageUDP()
    message.type = MessageTypes.command
    message.mac_address = mac
    message.device_name = socket.gethostname()
    message.is_encrypted = False

    command = Command()
    command.type = CommandTypes.rgb_stream
    command.sequence_number = 3004

    rgb_stream = CommandRGBStream()

    rgb_stream.leds_number = int(len(rgb_list) / 3) 


    rgb_stream.stream = list(rgb_list)

    command.content = rgb_stream
    message.content = command

    client_socket.sendto(message.to_bytes(), server_address)


def set_sequence(sequence):
    mac = uuid.getnode()


    message = MessageUDP()
    message.type = MessageTypes.command
    message.mac_address = mac
    message.device_name = socket.gethostname()
    message.is_encrypted = False

    command = Command()
    command.type = CommandTypes.sequence
    command.sequence_number = 3004

    sequence_command = CommandSequence()

    sequence_command.type = sequence

    command.content = sequence_command
    message.content = command

    client_socket.sendto(message.to_bytes(), server_address)


In [40]:
set_setting(0.2, 1)

In [ ]:
set_color([255,255,0])
set_sequence(2)

In [ ]:
from TickSystem.Transitions.EaseInOut import EaseInOut
from TickSystem import TickController


animation = EaseInOut()
animation.duration = 1
animation.starting_value = 0
animation.ending_value = 0.15
animation.on_update_callbacks.append(set_setting)

Tcontroller = TickController()

animation.start_tranition()

while True:
    animation.update()
    Tcontroller.wait_for_tick()
    
    if not animation.is_running:
        if animation.current_value <= 0:
            animation.start_tranition()
            continue

        if animation.current_value > 0:
            animation.reverse_transition()
            continue
        


In [ ]:
animation.update()


In [ ]:
set_color([255,255,0]*150)

In [ ]:
from TickSystem.Transitions.EaseInOut import EaseInOut
from TickSystem import TickController

LED_COUNT = 150
WAVE_WIDTH = 8

def custem_fun(value):
    # value already goes 0 → 1 → 0 (because of your animation logic)

    position = value * (LED_COUNT - 1)

    rgb = []

    for i in range(LED_COUNT):
        distance = abs(i - position)

        if distance < WAVE_WIDTH:
            intensity = 1 - (distance / WAVE_WIDTH)

            r = int(255 * intensity)
            g = 0
            b = int(255 * intensity)
        else:
            r, g, b = 0, 0, 0

        rgb.extend([r, g, b])

    set_color(rgb)


animation = EaseInOut()
animation.duration = 5
animation.starting_value = 0
animation.ending_value = 1
animation.on_update_callbacks.append(custem_fun)

Tcontroller = TickController()

animation.start_tranition()

while True:
    animation.update()
    Tcontroller.wait_for_tick()
    
    if not animation.is_running:
        if animation.current_value <= 0:
            animation.start_tranition()
            continue

        if animation.current_value > 0:
            animation.reverse_transition()
            continue
        